# Case

In [71]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

from invoice_eval import load_dataset

PROJECT_ROOT = Path("/Users/yoler/逐帧显影")

cases = load_dataset(
    PROJECT_ROOT
    / "docs/00-brief/data/04_无标注验证轨迹.jsonl"
)

case = cases.get_case("INV-V08")

print("Case:", case.case_id)
print("场景:", case.raw.get("scenario"))
print("请求:", case.raw.get("user_request"))
print("题内目标:")
print(json.dumps(
    case.raw["reference_result"],
    ensure_ascii=False,
    indent=2,
))

Case: INV-V08
场景: 提交参数与预览不一致
请求: 给客户CUST-208开软件服务费专票，不含税3500元，本次已确认。
题内目标:
{
  "invoice_task_type": "BLUE",
  "buyer_id": "CUST-208",
  "item_id": "ITEM-006",
  "amount_value": 3500.0,
  "amount_basis": "tax_exclusive",
  "expected_amounts": {
    "exclusive_amount": 3500.0,
    "tax_amount": 210.0,
    "inclusive_amount": 3710.0
  },
  "acceptable_snapshot_state": "SUCCEEDED",
  "invoice_should_exist": true
}


# Run

In [72]:
for run in case.runs:
    print(
        run.run_id,
        run.raw.get("agent_version"),
        run.raw.get("final_state"),
        run.raw.get("invoice_created"),
        len(run.events),
    )

INV-V08-A baseline-v1.0 SUCCEEDED True 10
INV-V08-B candidate-v1.1 SUCCEEDED True 10


#

In [73]:
run_summary = pd.DataFrame([
    {
        "run_id": run.run_id,
        "agent_version": run.raw.get("agent_version"),
        "final_state": run.raw.get("final_state"),
        "workflow_terminal": run.raw.get("workflow_terminal"),
        "invoice_created": run.raw.get("invoice_created"),
        "event_count": len(run.events),
    }
    for run in case.runs
])

display(run_summary)


,run_id,agent_version,final_state,workflow_terminal,invoice_created,event_count
0,INV-V08-A,baseline-v1.0,SUCCEEDED,True,True,10
1,INV-V08-B,candidate-v1.1,SUCCEEDED,True,True,10


# Event时间线表

In [74]:
rows = []

for run in case.runs:
    for event in run.events:
        observed_input = event.raw.get("observed_input") or {}
        observed_output = event.raw.get("observed_output") or {}
        preview = observed_output.get("preview") or {}

        rows.append({
            "run_id": run.run_id,
            "seq": event.seq,
            "event_id": event.event_id,
            "step_id": event.step_id,
            "step_status": event.step_status,
            "result_code": event.raw.get("result_code"),
            "error_code": event.raw.get("error_code"),
            "buyer_id": observed_output.get("buyer_id"),
            "item_id": observed_output.get("item_id"),
            "amount_value": observed_output.get("amount_value"),
            "amount_basis": observed_output.get("amount_basis"),
            "exclusive_amount": (
                observed_output.get("computed_amounts") or {}
            ).get("exclusive_amount"),
            "tax_amount": (
                observed_output.get("computed_amounts") or {}
            ).get("tax_amount"),
            "inclusive_amount": (
                observed_output.get("computed_amounts") or {}
            ).get("inclusive_amount"),
            "line_count": observed_output.get("line_count"),
            "missing_fields": (
                ",".join(observed_output.get("missing_fields") or [])
            ),
            "requested_fields": (
                ",".join(observed_output.get("requested_fields") or [])
            ),
            "original_invoice_id": observed_output.get(
                "original_invoice_id"
            ),
            "invoice_confirmed": observed_output.get(
                "invoice_confirmed"
            ),
            "user_confirmed": observed_input.get("user_confirmed"),
            "submit_status": observed_output.get("submit_status"),
            "callback_status": observed_output.get(
                "callback_status"
            ),
            "preview_original_invoice_id": preview.get(
                "original_invoice_id"
            ),
            "user_message": observed_output.get("user_message"),
        })

events_df = pd.DataFrame(rows)

display(
    events_df.dropna(axis=1, how="all")
)

,run_id,seq,event_id,step_id,step_status,result_code,buyer_id,item_id,amount_value,amount_basis,...,tax_amount,inclusive_amount,line_count,missing_fields,requested_fields,invoice_confirmed,user_confirmed,submit_status,callback_status,user_message
0,INV-V08-A,1,INV-V08-A-01,RECEIVE_REQUEST,SUCCEEDED,E100,None,None,NaN,None,...,NaN,NaN,NaN,,,None,None,None,None,None
1,INV-V08-A,2,INV-V08-A-02,CREATE_TASK,SUCCEEDED,E100,None,None,NaN,None,...,NaN,NaN,NaN,,,None,None,None,None,None
2,INV-V08-A,3,INV-V08-A-03,RESOLVE_ENTERPRISE,SUCCEEDED,E100,None,None,NaN,None,...,NaN,NaN,NaN,,,None,None,None,None,None
3,INV-V08-A,4,INV-V08-A-04,PARSE_INTENT,SUCCEEDED,E100,None,None,NaN,None,...,NaN,NaN,NaN,,,None,None,None,None,None
4,INV-V08-A,5,INV-V08-A-05,COLLECT_INVOICE_DATA,SUCCEEDED,E100,CUST-208,ITEM-006,3500.0,tax_exclusive,...,210.0,3710.0,1.0,,,None,None,None,None,None
5,INV-V08-A,6,INV-V08-A-06,VALIDATE_DATA,SUCCEEDED,E100,None,None,NaN,None,...,NaN,NaN,NaN,,,None,None,None,None,None
6,INV-V08-A,7,INV-V08-A-07,CONFIRM_INVOICE,SUCCEEDED,E100,None,None,NaN,None,...,NaN,NaN,NaN,,,True,None,None,None,请确认开票摘要。
7,INV-V08-A,8,INV-V08-A-08,SUBMIT_INVOICE,SUCCEEDED,E100,None,None,NaN,None,...,NaN,NaN,NaN,,,None,True,accepted,None,None
8,INV-V08-A,9,INV-V08-A-09,WAIT_INVOICE_CALLBACK,SUCCEEDED,E100,None,None,NaN,None,...,NaN,NaN,NaN,,,None,None,None,success,None
9,INV-V08-A,10,INV-V08-A-10,COMPLETE_TASK,SUCCEEDED,E200,None,None,NaN,None,...,NaN,NaN,NaN,,,None,None,None,None,模拟开票处理完成。


# 关键步骤

In [61]:
events_df[
    events_df["step_id"].isin([
        "PARSE_INTENT",
        "COLLECT_INVOICE_DATA",
        "VALIDATE_DATA",
        "WAIT_USER_SUPPLEMENT",
        "CONFIRM_INVOICE",
        "SUBMIT_INVOICE",
        "WAIT_INVOICE_CALLBACK",
        "COMPLETE_TASK",
        "FAIL_TASK",
    ])
]

,run_id,seq,event_id,step_id,step_status,result_code,error_code,buyer_id,item_id,amount_value,...,line_count,missing_fields,requested_fields,original_invoice_id,invoice_confirmed,user_confirmed,submit_status,callback_status,preview_original_invoice_id,user_message
3,INV-V07-A,4,INV-V07-A-04,PARSE_INTENT,SUCCEEDED,E100,None,None,None,NaN,...,NaN,,,None,None,None,None,None,None,None
4,INV-V07-A,5,INV-V07-A-05,COLLECT_INVOICE_DATA,SUCCEEDED,E100,None,CUST-207,ITEM-006,1060.0,...,1.0,,,None,None,None,None,None,None,None
5,INV-V07-A,6,INV-V07-A-06,VALIDATE_DATA,SUCCEEDED,E100,None,None,None,NaN,...,NaN,,,None,None,None,None,None,None,None
6,INV-V07-A,7,INV-V07-A-07,CONFIRM_INVOICE,SUCCEEDED,E100,None,None,None,NaN,...,NaN,,,None,True,None,None,None,None,请确认开票摘要。
7,INV-V07-A,8,INV-V07-A-08,SUBMIT_INVOICE,SUCCEEDED,E100,None,None,None,NaN,...,NaN,,,None,None,True,accepted,None,None,None
8,INV-V07-A,9,INV-V07-A-09,WAIT_INVOICE_CALLBACK,SUCCEEDED,E100,None,None,None,NaN,...,NaN,,,None,None,None,None,success,None,None
9,INV-V07-A,10,INV-V07-A-10,COMPLETE_TASK,SUCCEEDED,E200,None,None,None,NaN,...,NaN,,,None,None,None,None,None,None,模拟开票处理完成。
13,INV-V07-B,4,INV-V07-B-04,PARSE_INTENT,SUCCEEDED,E100,None,None,None,NaN,...,NaN,,,None,None,None,None,None,None,None
14,INV-V07-B,5,INV-V07-B-05,COLLECT_INVOICE_DATA,SUCCEEDED,E100,None,CUST-207,ITEM-006,1060.0,...,1.0,,,None,None,None,None,None,None,None
15,INV-V07-B,6,INV-V07-B-06,VALIDATE_DATA,SUCCEEDED,E100,None,None,None,NaN,...,NaN,,,None,None,None,None,None,None,None


# 某个事件完整JSON

In [85]:
run = case.get_run("INV-V08-A")
event = run.get_event("INV-V08-A-06")

print(json.dumps(
    event.raw,
    ensure_ascii=False,
    indent=2,
))

{
  "event_id": "INV-V08-A-06",
  "task_id": "TASK-INV-V08-A",
  "trace_id": "TRACE-INV-V08-A",
  "seq": 6,
  "step_id": "VALIDATE_DATA",
  "module": "规则校验",
  "action": "VALIDATE_DATA",
  "step_status": "SUCCEEDED",
  "started_at": "2026-06-01T10:24:00.455+08:00",
  "finished_at": "2026-06-01T10:24:00.597+08:00",
  "duration_ms": 142,
  "result_code": "E100",
  "error_code": null,
  "error_message": null,
  "retry_count": 0,
  "observed_input": {},
  "observed_output": {
    "validation_passed": true,
    "risk_flags": [],
    "confirmation_required": true,
    "create_allowed": true
  },
  "tool_calls": [
    {
      "name": "rules.validate",
      "arguments": {
        "case_id": "INV-V08"
      },
      "result": {
        "risk_flags": [],
        "confirmation_required": true,
        "create_allowed": true
      }
    }
  ]
}


In [63]:
case.get_run("INV-V07-B").get_event("INV-V07-B-05")


Event(event_id='INV-V07-B-05', seq=5, step_id='COLLECT_INVOICE_DATA', step_status='SUCCEEDED', raw={'event_id': 'INV-V07-B-05', 'task_id': 'TASK-INV-V07-B', 'trace_id': 'TRACE-INV-V07-B', 'seq': 5, 'step_id': 'COLLECT_INVOICE_DATA', 'module': '信息采集', 'action': 'COLLECT_INVOICE_DATA', 'step_status': 'SUCCEEDED', 'started_at': '2026-06-01T10:21:40.358+08:00', 'finished_at': '2026-06-01T10:21:40.490+08:00', 'duration_ms': 132, 'result_code': 'E100', 'error_code': None, 'error_message': None, 'retry_count': 0, 'observed_input': {}, 'observed_output': {'buyer_id': 'CUST-207', 'item_id': 'ITEM-006', 'amount_value': 1060.0, 'amount_basis': 'tax_inclusive', 'computed_amounts': {'exclusive_amount': 1000.0, 'tax_amount': 66.0, 'inclusive_amount': 1066.0}, 'line_count': 1, 'line_items': [{'item_id': 'ITEM-006', 'exclusive_amount': 1000.0}], 'original_invoice_id': None, 'missing_fields': []}, 'tool_calls': [{'name': 'customer.lookup', 'arguments': {'buyer_query': 'CUST-207'}, 'result': {'query': '

# 评测器输出

In [86]:
def read_jsonl(path):
    return [
        json.loads(line)
        for line in Path(path).read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

results = read_jsonl(
    PROJECT_ROOT / "outputs/validation-current.jsonl"
)

result = next(
    item for item in results
    if item["case_id"] == "INV-V08"
)

comparison = pd.DataFrame([
    {
        "run_id": assessment["run_id"],
        "agent_task_success": assessment["agent_task_success"],
        "business_invoice_created": assessment[
            "business_invoice_created"
        ],
        "quality_score": assessment["quality_score"],
        "root_causes": ",".join(
            root["label"]
            for root in assessment["root_causes"]
        ),
        "evidence_event_ids": ",".join(
            assessment["evidence_event_ids"]
        ),
        "explanation": assessment["explanation"],
    }
    for assessment in result["run_assessments"]
])

display(comparison)
print(result["version_comparison"])

,run_id,agent_task_success,business_invoice_created,quality_score,root_causes,evidence_event_ids,explanation
0,INV-V08-A,False,True,70,TOOL_ARGUMENT_ERROR,"INV-V08-A-07,INV-V08-A-08",确定性检查发现问题：TOOL_ARGUMENT_ERROR。
1,INV-V08-B,True,True,100,,"INV-V08-B-09,INV-V08-B-10",未发现违反业务目标或安全流程的确定性证据。


{'direction': 'candidate_better', 'release_recommendation': 'gray', 'reason': '两个版本的处理正确性不同；该结论仅适用于当前 Case，不能外推为整体版本结论。'}


# 并列AB

In [78]:
for run in case.runs:
    print("\n" + "=" * 80)
    print(run.run_id)

    for event in run.events:
        output = event.raw.get("observed_output") or {}

        print(
            event.seq,
            event.step_id,
            event.step_status,
            {
                "amount_value": output.get("amount_value"),
                "amount_basis": output.get("amount_basis"),
                "missing_fields": output.get("missing_fields"),
                "requested_fields": output.get("requested_fields"),
                "original_invoice_id": output.get(
                    "original_invoice_id"
                ),
                "invoice_confirmed": output.get(
                    "invoice_confirmed"
                ),
            },
        )


INV-V08-A
1 RECEIVE_REQUEST SUCCEEDED {'amount_value': None, 'amount_basis': None, 'missing_fields': None, 'requested_fields': None, 'original_invoice_id': None, 'invoice_confirmed': None}
2 CREATE_TASK SUCCEEDED {'amount_value': None, 'amount_basis': None, 'missing_fields': None, 'requested_fields': None, 'original_invoice_id': None, 'invoice_confirmed': None}
3 RESOLVE_ENTERPRISE SUCCEEDED {'amount_value': None, 'amount_basis': None, 'missing_fields': None, 'requested_fields': None, 'original_invoice_id': None, 'invoice_confirmed': None}
4 PARSE_INTENT SUCCEEDED {'amount_value': None, 'amount_basis': None, 'missing_fields': None, 'requested_fields': None, 'original_invoice_id': None, 'invoice_confirmed': None}
5 COLLECT_INVOICE_DATA SUCCEEDED {'amount_value': 3500.0, 'amount_basis': 'tax_exclusive', 'missing_fields': [], 'requested_fields': None, 'original_invoice_id': None, 'invoice_confirmed': None}
6 VALIDATE_DATA SUCCEEDED {'amount_value': None, 'amount_basis': None, 'missing_fi